# ACDADA — Notebook 10: FastAPI Backend

**Production REST API serving all ACDADA models & agents**

This notebook implements:
1. FastAPI application with all endpoints
2. Model loading & warm-up at startup
3. Pydantic schemas for request/response validation
4. WebSocket for real-time alerts & dashboard updates
5. Background task processing
6. Health checks, metrics, and logging
7. Script export for deployment

In [2]:
# ============================================================
# REFERENCE LINKS
# ============================================================
#
# FastAPI docs: https://fastapi.tiangolo.com/
# Pydantic v2:  https://docs.pydantic.dev/latest/
#
# This notebook creates the production API that loads
# every model from Notebooks 02-09 and exposes them
# to the React frontend.
#
# To run:
#   cd backend && uvicorn app.main:app --reload --port 8000
# ============================================================

## 1. Imports & Paths

In [3]:
import numpy as np
import torch
import torch.nn as nn
import json, time, uuid, asyncio, logging
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Any
from enum import Enum
from collections import deque

from fastapi import FastAPI, HTTPException, WebSocket, WebSocketDisconnect, BackgroundTasks, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn

# --------------- project paths ---------------
BASE_DIR   = Path(__file__).resolve().parent if '__file__' in dir() else Path('.')
MODELS_DIR = (BASE_DIR / '../models').resolve()
LOGS_DIR   = (BASE_DIR / '../logs').resolve()
LOGS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Models dir: {MODELS_DIR}')

Device: cpu  |  Models dir: D:\Projects\ACDADA\models


---
## 2. Pydantic Schemas (Request / Response)

In [4]:
# ===================== ENUMS =====================
class SeverityLevel(str, Enum):
    CRITICAL = 'critical'
    HIGH     = 'high'
    MEDIUM   = 'medium'
    LOW      = 'low'

class DecisionAction(str, Enum):
    BLOCK_AND_REDIRECT   = 'block_and_redirect'
    DEPLOY_DECEPTION     = 'deploy_deception'
    INCREASE_MONITORING  = 'increase_monitoring'
    OBSERVE              = 'observe'
    ALLOW                = 'allow'

# ============== REQUEST SCHEMAS =================
class AnalyzeRequest(BaseModel):
    """Full pipeline analysis — sent from the React dashboard."""
    features: List[float] = Field(..., description='Numeric feature vector (from preprocessor)')
    label: int = Field(-1, description='Ground-truth label (-1 = unknown, for online evaluation)')
    flow_id: Optional[str] = Field(None, description='Optional caller-assigned flow ID')

class ThreatDetectRequest(BaseModel):
    features: List[float]

class AnomalyDetectRequest(BaseModel):
    features: List[float]

class ClassifyRequest(BaseModel):
    features: List[float]

class ThreatIntelRequest(BaseModel):
    description: str = Field(..., min_length=5)
    k: int = Field(5, ge=1, le=20)
    category_filter: Optional[str] = None
    severity_filter: Optional[str] = None

class AttackProfileRequest(BaseModel):
    indicators: List[str] = Field(..., min_length=1)

class DeceptionActionRequest(BaseModel):
    observation: List[float] = Field(..., description='Env observation vector (95 floats)')
    model_type: str = Field('ppo', description='ppo | ppo_curriculum | dqn')

# ============== RESPONSE SCHEMAS ================
class ThreatDetectResponse(BaseModel):
    is_threat: bool
    confidence: float
    model_used: str
    latency_ms: float

class AnomalyDetectResponse(BaseModel):
    is_anomaly: bool
    anomaly_score: float
    threshold: float
    method_scores: Dict[str, float]
    latency_ms: float

class ClassifyResponse(BaseModel):
    attack_type: str
    attack_type_id: int
    confidence: float
    probabilities: Dict[str, float]
    latency_ms: float

class ThreatIntelResponse(BaseModel):
    likely_category: str
    likely_severity: str
    confidence: float
    top_tags: List[str]
    similar_threats: List[Dict[str, Any]]
    recommendations: List[str]

class DeceptionActionResponse(BaseModel):
    action_id: int
    action_name: str
    model_type: str
    latency_ms: float

class PipelineResponse(BaseModel):
    """Full pipeline result — consumed by the React dashboard."""
    flow_id: str
    timestamp: str
    is_threat: bool
    threat_confidence: float
    is_anomaly: bool
    anomaly_score: float
    attack_type: Optional[str] = None
    attack_confidence: Optional[float] = None
    severity: SeverityLevel
    decision: DecisionAction
    deception_action: Optional[str] = None
    recommendations: List[str] = []
    consensus_score: float = 0.0
    processing_time_ms: float
    agent_outputs: Dict[str, Any] = {}

class SystemStatusResponse(BaseModel):
    overall_health: float
    n_agents: int
    n_alerts: int
    agents: Dict[str, Any]
    uptime_seconds: float
    total_events_processed: int
    recent_events: List[Dict[str, Any]] = []

class HealthResponse(BaseModel):
    status: str
    models_loaded: Dict[str, bool]
    device: str
    uptime_seconds: float

print('Schemas defined.')

Schemas defined.


---
## 3. Model Registry — Load All Agents at Startup

Each loader mirrors the function from its source notebook.

In [5]:
# =======================================================
#  Thin model wrappers so the API code stays clean.
#  We re-declare lightweight class stubs that match the
#  architectures saved in Notebooks 02-09.
# =======================================================

# ---- Notebook 02: Threat Detector architectures ----
class CNNThreatDetector(nn.Module):
    def __init__(self, n_features, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(n_features, 128)
        self.convs = nn.Sequential(
            nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128),
            nn.Conv1d(128, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 2))
    def forward(self, x):
        x = self.proj(x).unsqueeze(1)
        x = self.convs(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)

class LSTMThreatDetector(nn.Module):
    def __init__(self, n_features, hidden_size=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(n_features, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(hidden_size*2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 2))
    def forward(self, x):
        x = self.proj(x).unsqueeze(1).expand(-1, 8, -1)
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (out * w).sum(dim=1)
        return self.classifier(ctx)

class HybridCNNLSTMDetector(nn.Module):
    def __init__(self, n_features, cnn_channels=64, lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(n_features, 128)
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(cnn_channels),
            nn.Conv1d(cnn_channels, cnn_channels, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(cnn_channels))
        self.lstm = nn.LSTM(cnn_channels, lstm_hidden, 2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(lstm_hidden*2, 1)
        self.res_proj = nn.Linear(cnn_channels, lstm_hidden*2)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden*2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 2))
    def forward(self, x):
        x = self.proj(x).unsqueeze(1)
        c = self.cnn(x)
        c_t = c.permute(0, 2, 1)
        out, _ = self.lstm(c_t)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (out * w).sum(1)
        res = self.res_proj(c.mean(dim=-1))
        return self.classifier(ctx + res)

THREAT_MODEL_MAP = {
    'CNNThreatDetector': CNNThreatDetector,
    'LSTMThreatDetector': LSTMThreatDetector,
    'HybridCNNLSTMDetector': HybridCNNLSTMDetector,
}

# ---- Notebook 03: Anomaly Detector architectures ----
class DeepAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=16, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, n_features))
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def anomaly_score(self, x):
        return ((self.forward(x) - x) ** 2).mean(dim=1)

class VAEAnomalyDetector(nn.Module):
    def __init__(self, n_features, latent_dim=16, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU())
        self.fc_mu  = nn.Linear(64, latent_dim)
        self.fc_var = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, n_features))
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_var(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)
    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decoder(z), mu, lv
    def anomaly_score(self, x):
        recon, mu, lv = self.forward(x)
        recon_loss = ((recon - x)**2).mean(dim=1)
        kl = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum(dim=1)
        return recon_loss + 0.001 * kl

# ---- Notebook 04: Attack Classifier architecture ----
class DNNAttackClassifier(nn.Module):
    def __init__(self, n_features, n_classes, dropout=0.3):
        super().__init__()
        self.blocks = nn.Sequential(
            self._block(n_features, 256, dropout),
            self._block(256, 256, dropout),
            self._block(256, 128, dropout),
            self._block(128, 64,  dropout),
        )
        self.head = nn.Linear(64, n_classes)
    @staticmethod
    def _block(inp, out, dr):
        return nn.Sequential(
            nn.Linear(inp, out), nn.BatchNorm1d(out), nn.ReLU(), nn.Dropout(dr))
    def forward(self, x):
        return self.head(self.blocks(x))

# ---- Notebook 06: DQN architecture ----
class DuelingDQN(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.LayerNorm(hidden))
        self.value = nn.Sequential(nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, n_actions))
    def forward(self, x):
        f = self.features(x)
        v = self.value(f)
        a = self.advantage(f)
        return v + a - a.mean(dim=-1, keepdim=True)

print('Architecture stubs declared.')

Architecture stubs declared.


In [6]:
import joblib

class ModelRegistry:
    """
    Loads every model once at startup and serves them to route handlers.
    """

    def __init__(self, models_dir: Path):
        self.dir = models_dir
        self.loaded: Dict[str, bool] = {}

        # placeholders
        self.threat_model = None
        self.threat_meta  = {}

        self.ae_model  = None
        self.vae_model = None
        self.if_model  = None
        self.anomaly_ensemble_cfg = {}

        self.xgb_model     = None
        self.dnn_model     = None
        self.cls_ensemble   = {}
        self.label_encoder  = None
        self.class_names    = []

        self.deception_ppo       = None
        self.deception_ppo_curr  = None
        self.deception_dqn       = None

        self.threat_intel_engine = None

    # ------------- loaders --------------------------------
    def load_all(self):
        self._load_threat_detector()
        self._load_anomaly_detector()
        self._load_attack_classifier()
        self._load_deception_agents()
        self._load_threat_intel()
        print(f'\nModel registry status: {self.loaded}')

    # -- 02 Threat detection ------
    def _load_threat_detector(self):
        tag = 'threat_detector'
        p = self.dir / 'threat_detection' / 'best_threat_detector.pth'
        try:
            ckpt = torch.load(p, map_location=DEVICE, weights_only=False)
            cls = THREAT_MODEL_MAP[ckpt['model_class']]
            model = cls(n_features=ckpt['n_features']).to(DEVICE)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()
            self.threat_model = model
            self.threat_meta  = ckpt
            self.loaded[tag] = True
            print(f'  ✓ {tag} ({ckpt["model_class"]}, {ckpt["n_features"]} features)')
        except Exception as e:
            self.loaded[tag] = False
            print(f'  ✗ {tag}: {e}')

    # -- 03 Anomaly detection ------
    def _load_anomaly_detector(self):
        tag = 'anomaly_detector'
        ad = self.dir / 'anomaly_detection'
        try:
            # Autoencoder
            ae_ckpt = torch.load(ad / 'autoencoder.pth', map_location=DEVICE, weights_only=False)
            self.ae_model = DeepAutoencoder(ae_ckpt['n_features'], ae_ckpt['latent_dim']).to(DEVICE)
            self.ae_model.load_state_dict(ae_ckpt['model_state_dict']); self.ae_model.eval()

            # VAE
            vae_ckpt = torch.load(ad / 'vae.pth', map_location=DEVICE, weights_only=False)
            self.vae_model = VAEAnomalyDetector(vae_ckpt['n_features'], vae_ckpt['latent_dim']).to(DEVICE)
            self.vae_model.load_state_dict(vae_ckpt['model_state_dict']); self.vae_model.eval()

            # Isolation Forest
            if_data = joblib.load(ad / 'isolation_forest.joblib')
            self.if_model = if_data['model'] if isinstance(if_data, dict) else if_data

            # Ensemble config
            self.anomaly_ensemble_cfg = joblib.load(ad / 'ensemble_config.joblib')

            self.loaded[tag] = True
            print(f'  ✓ {tag} (AE + VAE + IF + Ensemble)')
        except Exception as e:
            self.loaded[tag] = False
            print(f'  ✗ {tag}: {e}')

    # -- 04 Attack classification ------
    def _load_attack_classifier(self):
        tag = 'attack_classifier'
        ac = self.dir / 'attack_classification'
        try:
            # XGBoost
            xgb_data = joblib.load(ac / 'xgboost_classifier.joblib')
            self.xgb_model = xgb_data['model'] if isinstance(xgb_data, dict) else xgb_data

            # DNN
            dnn_ckpt = torch.load(ac / 'dnn_classifier.pth', map_location=DEVICE, weights_only=False)
            self.dnn_model = DNNAttackClassifier(
                dnn_ckpt['n_features'], dnn_ckpt['n_classes']).to(DEVICE)
            self.dnn_model.load_state_dict(dnn_ckpt['model_state_dict']); self.dnn_model.eval()

            self.cls_ensemble  = joblib.load(ac / 'ensemble_config.joblib')
            self.label_encoder = joblib.load(ac / 'label_encoder.joblib')
            self.class_names   = list(dnn_ckpt.get('class_names', []))

            self.loaded[tag] = True
            print(f'  ✓ {tag} (XGB + DNN, {len(self.class_names)} classes)')
        except Exception as e:
            self.loaded[tag] = False
            print(f'  ✗ {tag}: {e}')

    # -- 06 Deception RL agents ------
    def _load_deception_agents(self):
        tag = 'deception_agent'
        da = self.dir / 'deception_agent'
        try:
            from stable_baselines3 import PPO
            # We don't need the env for predict-only mode
            self.deception_ppo = PPO.load(da / 'ppo_deception', device=DEVICE)
            if (da / 'ppo_curriculum.zip').exists():
                self.deception_ppo_curr = PPO.load(da / 'ppo_curriculum', device=DEVICE)

            # DQN
            dqn_path = da / 'dqn_final.pth'
            if dqn_path.exists():
                net = DuelingDQN(95, 9).to(DEVICE)
                net.load_state_dict(torch.load(dqn_path, map_location=DEVICE, weights_only=True))
                net.eval()
                self.deception_dqn = net

            self.loaded[tag] = True
            print(f'  ✓ {tag} (PPO + DQN)')
        except Exception as e:
            self.loaded[tag] = False
            print(f'  ✗ {tag}: {e}')

    # -- 07 Threat Intel memory ------
    def _load_threat_intel(self):
        tag = 'threat_intel'
        ti = self.dir / 'threat_intel'
        try:
            import faiss
            with open(ti / 'embedder_config.json') as f:
                ecfg = json.load(f)

            # --- embedder (lightweight re-impl) ---
            class _Embedder:
                def __init__(self, cfg, models_dir):
                    self.fallback = cfg.get('fallback', True)
                    self.embed_dim = cfg['embed_dim']
                    self.model = None
                    if not self.fallback:
                        try:
                            from sentence_transformers import SentenceTransformer
                            self.model = SentenceTransformer(cfg['model_name'])
                        except ImportError:
                            self.fallback = True
                    if self.fallback:
                        self._vec = joblib.load(models_dir / 'tfidf_vectorizer.joblib')
                        self._svd = joblib.load(models_dir / 'tfidf_svd.joblib')
                def embed_query(self, text):
                    if not self.fallback:
                        return self.model.encode([text], normalize_embeddings=True).astype(np.float32)[0]
                    from sklearn.preprocessing import normalize
                    tf = self._vec.transform([text])
                    return normalize(self._svd.transform(tf), 'l2').astype(np.float32)[0]

            embedder = _Embedder(ecfg, ti)

            # --- FAISS index ---
            faiss_dir = ti / 'faiss'
            index = faiss.read_index(str(faiss_dir / 'faiss_index.bin'))
            with open(faiss_dir / 'metadata.json') as f:
                meta = json.load(f)

            class _Intel:
                """ Minimal TI engine for the API layer. """
                def __init__(self, idx, meta, emb):
                    self.index = idx
                    self.meta  = meta
                    self.emb   = emb
                    self.idx2id = {int(v): k for k, v in meta['id_to_idx'].items()}
                    self.records = meta['records']

                def query(self, text, k=5):
                    q = self.emb.embed_query(text).reshape(1, -1).astype(np.float32)
                    faiss.normalize_L2(q)
                    scores, ids = self.index.search(q, k)
                    results = []
                    from collections import Counter
                    cats, sevs, tags_c = Counter(), Counter(), Counter()
                    for sc, i in zip(scores[0], ids[0]):
                        if i < 0: continue
                        rid = self.idx2id.get(i)
                        if rid and rid in self.records:
                            r = self.records[rid]
                            results.append({'text': r['text'], 'category': r['category'],
                                            'severity': r['severity'], 'similarity': float(sc),
                                            'tags': r.get('tags',[])})
                            cats[r['category']] += 1; sevs[r['severity']] += 1
                            for t in r.get('tags',[]): tags_c[t] += 1
                    cat = cats.most_common(1)[0][0] if cats else 'unknown'
                    sev = sevs.most_common(1)[0][0] if sevs else 'medium'
                    return {'likely_category': cat, 'likely_severity': sev,
                            'confidence': results[0]['similarity'] if results else 0,
                            'top_tags': [t for t,_ in tags_c.most_common(5)],
                            'similar_threats': results,
                            'recommendations': self._recs(cat, sev)}

                @staticmethod
                def _recs(cat, sev):
                    m = {'ddos':['Rate-limit traffic','Activate scrubbing','Deploy honeypot'],
                         'malware':['Isolate host','Scan for IOCs','Update signatures'],
                         'phishing':['Block sender','Rotate creds','Monitor logins'],
                         'brute_force':['Deploy SSH honeypot','Enable MFA','Block source'],
                         'injection':['Deploy WAF rule','Patch endpoint','Deploy decoy DB'],
                         'apt':['Full sweep','Deploy advanced honeypots','Max monitoring'],
                         'botnet':['Isolate hosts','Block C2','Scan network'],
                         'lateral_movement':['Deploy honeytokens','Segment network','Increase monitoring'],
                         'exfiltration':['Block outbound','Deploy canaries','Activate DLP']}
                    recs = m.get(cat, ['Increase monitoring','Alert SOC'])
                    if sev in ('critical','high'):
                        recs.insert(0, 'PRIORITY: Immediate escalation')
                    return recs

            self.threat_intel_engine = _Intel(index, meta, embedder)
            self.loaded[tag] = True
            print(f'  ✓ {tag} (FAISS {index.ntotal} vectors, dim={ecfg["embed_dim"]})')
        except Exception as e:
            self.loaded[tag] = False
            print(f'  ✗ {tag}: {e}')


print('ModelRegistry defined.')

ModelRegistry defined.


---
## 4. Inference Helpers

In [7]:
# =====================================================
#  Stateless inference functions used by route handlers
# =====================================================

def infer_threat(reg: ModelRegistry, features: np.ndarray) -> dict:
    """Binary threat detection."""
    t0 = time.perf_counter()
    x = torch.FloatTensor(features).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = reg.threat_model(x)
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    return {'is_threat': prob > 0.5, 'confidence': round(prob, 5),
            'model_used': reg.threat_meta.get('model_class', 'unknown'),
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2)}


def infer_anomaly(reg: ModelRegistry, features: np.ndarray) -> dict:
    """Ensemble anomaly detection."""
    t0 = time.perf_counter()
    x_t = torch.FloatTensor(features).unsqueeze(0).to(DEVICE)
    x_np = features.reshape(1, -1)

    with torch.no_grad():
        ae_score  = reg.ae_model.anomaly_score(x_t).item()
        vae_score = reg.vae_model.anomaly_score(x_t).item()
    if_score = float(-reg.if_model.decision_function(x_np)[0])

    # normalise with saved scalers
    cfg = reg.anomaly_ensemble_cfg
    scalers = cfg.get('scalers', {})
    weights = cfg.get('weights', {'ae': 0.33, 'vae': 0.33, 'if': 0.34})
    threshold = cfg.get('threshold', 0.5)

    def _norm(val, key):
        s = scalers.get(key)
        if s is not None:
            return float(np.clip(s.transform([[val]])[0, 0], 0, 1))
        return float(np.clip(val, 0, 1))

    scores = {'ae': _norm(ae_score, 'ae'), 'vae': _norm(vae_score, 'vae'), 'if': _norm(if_score, 'if')}
    ensemble = sum(scores[k] * weights.get(k, 0.33) for k in scores)

    return {'is_anomaly': ensemble > threshold, 'anomaly_score': round(ensemble, 5),
            'threshold': threshold, 'method_scores': {k: round(v, 5) for k, v in scores.items()},
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2)}


def infer_classify(reg: ModelRegistry, features: np.ndarray) -> dict:
    """Multi-class attack classification."""
    t0 = time.perf_counter()
    x_np = features.reshape(1, -1)
    x_t  = torch.FloatTensor(features).unsqueeze(0).to(DEVICE)

    # XGBoost
    xgb_prob = reg.xgb_model.predict_proba(x_np)[0]

    # DNN
    with torch.no_grad():
        dnn_prob = torch.softmax(reg.dnn_model(x_t), dim=1)[0].cpu().numpy()

    # Ensemble (soft vote)
    w = reg.cls_ensemble.get('weights', {'xgb': 0.5, 'dnn': 0.5})
    combined = w.get('xgb', 0.5) * xgb_prob + w.get('dnn', 0.5) * dnn_prob
    pred_id  = int(combined.argmax())
    names    = reg.class_names or [str(i) for i in range(len(combined))]
    prob_map = {names[i]: round(float(combined[i]), 5) for i in range(len(combined))}

    return {'attack_type': names[pred_id], 'attack_type_id': pred_id,
            'confidence': round(float(combined[pred_id]), 5),
            'probabilities': prob_map,
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2)}


def infer_deception(reg: ModelRegistry, obs: np.ndarray, model_type: str = 'ppo') -> dict:
    """Deception RL action selection."""
    ACTION_NAMES = ['observe','activate_hp_0','activate_hp_1','activate_hp_2',
                    'activate_hp_3','deactivate_all','redirect_to_hp',
                    'increase_monitoring','block_attacker']
    t0 = time.perf_counter()
    if model_type == 'dqn' and reg.deception_dqn is not None:
        with torch.no_grad():
            t = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
            action = int(reg.deception_dqn(t).argmax(1).item())
    elif model_type == 'ppo_curriculum' and reg.deception_ppo_curr is not None:
        action, _ = reg.deception_ppo_curr.predict(obs, deterministic=True)
        action = int(action)
    elif reg.deception_ppo is not None:
        action, _ = reg.deception_ppo.predict(obs, deterministic=True)
        action = int(action)
    else:
        action = 0
    return {'action_id': action, 'action_name': ACTION_NAMES[action],
            'model_type': model_type,
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2)}


def run_full_pipeline(reg: ModelRegistry, features: np.ndarray,
                      label: int = -1, flow_id: str = None) -> dict:
    """
    Orchestrated pipeline — mirrors Notebook 09 logic.
    detect → classify → enrich → decide → act
    """
    t0 = time.perf_counter()
    fid = flow_id or str(uuid.uuid4())[:12]
    agent_out = {}

    # 1) Threat detection
    td = infer_threat(reg, features) if reg.loaded.get('threat_detector') else \
         {'is_threat': False, 'confidence': 0, 'model_used': 'none', 'latency_ms': 0}
    agent_out['threat_detection'] = td

    # 2) Anomaly detection
    ad = infer_anomaly(reg, features) if reg.loaded.get('anomaly_detector') else \
         {'is_anomaly': False, 'anomaly_score': 0, 'threshold': 0.5, 'method_scores': {}, 'latency_ms': 0}
    agent_out['anomaly_detection'] = ad

    # 3) If suspicious → classify + enrich
    cl = None; ti = None; dec_action = None
    suspicious = td['is_threat'] or ad['is_anomaly']

    if suspicious and reg.loaded.get('attack_classifier'):
        cl = infer_classify(reg, features)
        agent_out['classification'] = cl

    if suspicious and reg.loaded.get('threat_intel'):
        desc = f"{cl['attack_type']} attack detected" if cl else 'suspicious network activity'
        ti = reg.threat_intel_engine.query(desc, k=3)
        agent_out['threat_intel'] = ti

    # 4) Consensus & severity
    consensus = 0.0
    if suspicious:
        consensus = 0.4 * td['confidence'] + 0.3 * ad['anomaly_score']
        if cl:
            consensus += 0.3 * cl['confidence']

    if   consensus > 0.8: severity = SeverityLevel.CRITICAL
    elif consensus > 0.6: severity = SeverityLevel.HIGH
    elif consensus > 0.3: severity = SeverityLevel.MEDIUM
    else:                 severity = SeverityLevel.LOW

    # 5) Decision
    if not suspicious:
        decision = DecisionAction.ALLOW
    elif severity == SeverityLevel.CRITICAL:
        decision = DecisionAction.BLOCK_AND_REDIRECT
    elif severity == SeverityLevel.HIGH:
        decision = DecisionAction.DEPLOY_DECEPTION
    elif severity == SeverityLevel.MEDIUM:
        decision = DecisionAction.INCREASE_MONITORING
    else:
        decision = DecisionAction.OBSERVE

    # 6) Deception action (placeholder obs — real env would supply this)
    if suspicious and reg.loaded.get('deception_agent'):
        dummy_obs = np.zeros(95, dtype=np.float32)
        dec = infer_deception(reg, dummy_obs, 'ppo')
        dec_action = dec['action_name']
        agent_out['deception'] = dec

    return {
        'flow_id': fid,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'is_threat': td['is_threat'],
        'threat_confidence': td['confidence'],
        'is_anomaly': ad['is_anomaly'],
        'anomaly_score': ad['anomaly_score'],
        'attack_type': cl['attack_type'] if cl else None,
        'attack_confidence': cl['confidence'] if cl else None,
        'severity': severity,
        'decision': decision,
        'deception_action': dec_action,
        'recommendations': ti.get('recommendations', []) if ti else [],
        'consensus_score': round(consensus, 5),
        'processing_time_ms': round((time.perf_counter() - t0) * 1000, 2),
        'agent_outputs': agent_out,
    }

print('Inference helpers defined.')

Inference helpers defined.


---
## 5. FastAPI Application

In [8]:
# =====================================================
#  Application factory
# =====================================================

START_TIME = time.time()
EVENT_LOG: deque = deque(maxlen=500)  # recent events for dashboard
TOTAL_EVENTS = 0


def create_app(models_dir: Path = MODELS_DIR) -> FastAPI:
    app = FastAPI(
        title='ACDADA — Autonomous Cyber Deception API',
        description='Multi-agent RL-driven cyber defense REST API',
        version='1.0.0',
    )

    # CORS — allow React dev server
    app.add_middleware(
        CORSMiddleware,
        allow_origins=['http://localhost:3000', 'http://localhost:5173', '*'],
        allow_credentials=True,
        allow_methods=['*'],
        allow_headers=['*'],
    )

    # ---------- startup ----------
    registry = ModelRegistry(models_dir)

    @app.on_event('startup')
    async def startup():
        global START_TIME
        START_TIME = time.time()
        print('\n🚀 Loading models...')
        registry.load_all()
        print('✓ ACDADA API ready.\n')

    # --------- WebSocket manager ----------
    class WSManager:
        def __init__(self):
            self.connections: List[WebSocket] = []
        async def connect(self, ws: WebSocket):
            await ws.accept(); self.connections.append(ws)
        def disconnect(self, ws: WebSocket):
            self.connections.remove(ws)
        async def broadcast(self, data: dict):
            dead = []
            for ws in self.connections:
                try: await ws.send_json(data)
                except: dead.append(ws)
            for ws in dead: self.connections.remove(ws)

    ws_mgr = WSManager()

    # ============================================================
    #  ROUTES
    # ============================================================

    # ----- Health -----
    @app.get('/health', response_model=HealthResponse, tags=['System'])
    async def health():
        return HealthResponse(
            status='ok',
            models_loaded=registry.loaded,
            device=str(DEVICE),
            uptime_seconds=round(time.time() - START_TIME, 1),
        )

    # ----- System status (dashboard home) -----
    @app.get('/status', response_model=SystemStatusResponse, tags=['System'])
    async def system_status():
        n_loaded = sum(v for v in registry.loaded.values())
        return SystemStatusResponse(
            overall_health=round(n_loaded / max(len(registry.loaded), 1) * 100, 1),
            n_agents=len(registry.loaded),
            n_alerts=sum(1 for e in EVENT_LOG if e.get('severity') in ('critical','high')),
            agents={k: {'loaded': v} for k, v in registry.loaded.items()},
            uptime_seconds=round(time.time() - START_TIME, 1),
            total_events_processed=TOTAL_EVENTS,
            recent_events=list(EVENT_LOG)[-20:],
        )

    # ----- Full pipeline -----
    @app.post('/analyze', response_model=PipelineResponse, tags=['Pipeline'])
    async def analyze(req: AnalyzeRequest, background_tasks: BackgroundTasks):
        global TOTAL_EVENTS
        features = np.array(req.features, dtype=np.float32)
        result = run_full_pipeline(registry, features, req.label, req.flow_id)
        TOTAL_EVENTS += 1

        # store + broadcast
        evt = {k: (v.value if isinstance(v, Enum) else v)
               for k, v in result.items() if k != 'agent_outputs'}
        EVENT_LOG.append(evt)
        background_tasks.add_task(ws_mgr.broadcast, evt)

        return PipelineResponse(**result)

    # ----- Individual agent endpoints -----
    @app.post('/detect/threat', response_model=ThreatDetectResponse, tags=['Agents'])
    async def detect_threat(req: ThreatDetectRequest):
        if not registry.loaded.get('threat_detector'):
            raise HTTPException(503, 'Threat detector not loaded')
        return ThreatDetectResponse(**infer_threat(registry, np.array(req.features, dtype=np.float32)))

    @app.post('/detect/anomaly', response_model=AnomalyDetectResponse, tags=['Agents'])
    async def detect_anomaly(req: AnomalyDetectRequest):
        if not registry.loaded.get('anomaly_detector'):
            raise HTTPException(503, 'Anomaly detector not loaded')
        return AnomalyDetectResponse(**infer_anomaly(registry, np.array(req.features, dtype=np.float32)))

    @app.post('/classify', response_model=ClassifyResponse, tags=['Agents'])
    async def classify_attack(req: ClassifyRequest):
        if not registry.loaded.get('attack_classifier'):
            raise HTTPException(503, 'Attack classifier not loaded')
        return ClassifyResponse(**infer_classify(registry, np.array(req.features, dtype=np.float32)))

    @app.post('/deception/action', response_model=DeceptionActionResponse, tags=['Agents'])
    async def deception_action(req: DeceptionActionRequest):
        if not registry.loaded.get('deception_agent'):
            raise HTTPException(503, 'Deception agent not loaded')
        return DeceptionActionResponse(
            **infer_deception(registry, np.array(req.observation, dtype=np.float32), req.model_type))

    # ----- Threat Intelligence -----
    @app.post('/intel/query', response_model=ThreatIntelResponse, tags=['Intelligence'])
    async def intel_query(req: ThreatIntelRequest):
        if not registry.loaded.get('threat_intel'):
            raise HTTPException(503, 'Threat intel engine not loaded')
        return ThreatIntelResponse(**registry.threat_intel_engine.query(req.description, req.k))

    @app.post('/intel/profile', tags=['Intelligence'])
    async def intel_profile(req: AttackProfileRequest):
        if not registry.loaded.get('threat_intel'):
            raise HTTPException(503, 'Threat intel engine not loaded')
        from collections import Counter
        cats, sevs, all_recs = Counter(), Counter(), []
        for ind in req.indicators:
            r = registry.threat_intel_engine.query(ind, k=3)
            cats[r['likely_category']] += 1
            sevs[r['likely_severity']] += 1
            all_recs.extend(r['recommendations'])
        return {'n_indicators': len(req.indicators),
                'attack_categories': dict(cats.most_common()),
                'overall_severity': sevs.most_common(1)[0][0] if sevs else 'medium',
                'recommendations': list(dict.fromkeys(all_recs))[:10]}

    # ----- Event log -----
    @app.get('/events', tags=['Events'])
    async def get_events(limit: int = Query(50, ge=1, le=500)):
        return list(EVENT_LOG)[-limit:]

    # ----- WebSocket — real-time dashboard feed -----
    @app.websocket('/ws/events')
    async def ws_events(ws: WebSocket):
        await ws_mgr.connect(ws)
        try:
            while True:
                await ws.receive_text()  # keep-alive
        except WebSocketDisconnect:
            ws_mgr.disconnect(ws)

    return app


app = create_app()
print('FastAPI app created.  Endpoints:')
for r in app.routes:
    if hasattr(r, 'methods'):
        print(f'  {list(r.methods)[0]:6s} {r.path}')

FastAPI app created.  Endpoints:
  GET    /openapi.json
  GET    /docs
  GET    /docs/oauth2-redirect
  GET    /redoc
  GET    /health
  GET    /status
  POST   /analyze
  POST   /detect/threat
  POST   /detect/anomaly
  POST   /classify
  POST   /deception/action
  POST   /intel/query
  POST   /intel/profile
  GET    /events


C:\Users\HP\AppData\Local\Temp\ipykernel_13440\159510643.py:29: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


---
## 6. Export to Standalone `backend/` Package

Write the application as proper Python files that can be run with `uvicorn`.

In [ ]:
import textwrap, os

BACKEND_ROOT = Path('../backend')
APP_DIR      = BACKEND_ROOT / 'app'
for d in [APP_DIR, APP_DIR / 'api', APP_DIR / 'core', APP_DIR / 'models', APP_DIR / 'schemas']:
    d.mkdir(parents=True, exist_ok=True)

# ---- requirements.txt ----
(BACKEND_ROOT / 'requirements.txt').write_text(textwrap.dedent("""\
    fastapi>=0.109.0
    uvicorn[standard]>=0.27.0
    pydantic>=2.5.0
    numpy>=1.24.0
    torch>=2.1.0
    scikit-learn>=1.3.0
    xgboost>=2.0.0
    joblib>=1.3.0
    faiss-cpu>=1.7.0
    stable-baselines3>=2.2.0
    sentence-transformers>=2.2.0
    python-multipart>=0.0.6
    websockets>=12.0
"""), encoding='utf-8')

# ---- app/__init__.py ----
(APP_DIR / '__init__.py').write_text('', encoding='utf-8')
(APP_DIR / 'api' / '__init__.py').write_text('', encoding='utf-8')
(APP_DIR / 'core' / '__init__.py').write_text('', encoding='utf-8')
(APP_DIR / 'models' / '__init__.py').write_text('', encoding='utf-8')
(APP_DIR / 'schemas' / '__init__.py').write_text('', encoding='utf-8')

print('Backend scaffold created.')
for root, dirs, files in os.walk(BACKEND_ROOT):
    level = root.replace(str(BACKEND_ROOT), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{Path(root).name}/')
    for f in files:
        print(f'{indent}  {f}')

Backend scaffold created.
backend/
  requirements.txt
  app/
    __init__.py
    api/
      __init__.py
    core/
      __init__.py
    models/
      __init__.py
    schemas/
      __init__.py


In [ ]:
# ---- app/core/config.py ----
(APP_DIR / 'core' / 'config.py').write_text(textwrap.dedent('''\
    from pathlib import Path
    import torch

    BASE_DIR   = Path(__file__).resolve().parent.parent.parent   # backend/
    MODELS_DIR = (BASE_DIR.parent / "models").resolve()
    LOGS_DIR   = (BASE_DIR.parent / "logs").resolve()
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CORS_ORIGINS = [
        "http://localhost:3000",
        "http://localhost:5173",
    ]
'''), encoding='utf-8')

print('core/config.py written.')

core/config.py written.


In [ ]:
# ---- app/schemas/requests.py ----
(APP_DIR / 'schemas' / 'requests.py').write_text(textwrap.dedent('''\
    from pydantic import BaseModel, Field
    from typing import List, Optional

    class AnalyzeRequest(BaseModel):
        features: List[float] = Field(..., description="Numeric feature vector")
        label: int = Field(-1, description="Ground-truth label (-1 = unknown)")
        flow_id: Optional[str] = None
        threat_model: str = Field("mlp", description="Model for threat detection")
        anomaly_threshold: float = Field(0.5, ge=0, le=1)

    class ThreatDetectRequest(BaseModel):
        features: List[float]
        model: str = Field("mlp", description="mlp, ensemble, etc.")

    class AnomalyDetectRequest(BaseModel):
        features: List[float]
        threshold: float = Field(0.5, ge=0, le=1)

    class ClassifyRequest(BaseModel):
        features: List[float]
        model: str = Field("dnn", description="dnn, xgboost, ensemble")

    class ThreatIntelRequest(BaseModel):
        description: str = Field(..., min_length=5)
        top_k: int = Field(5, ge=1, le=20)
        category_filter: Optional[str] = None
        severity_filter: Optional[str] = None

    class AttackProfileRequest(BaseModel):
        indicators: List[str] = Field(..., min_length=1)

    class DeceptionActionRequest(BaseModel):
        threat_level: float = Field(0.5, ge=0, le=1)
        attack_type: Optional[str] = None
        is_detected: bool = False
'''), encoding='utf-8')

# ---- app/schemas/responses.py ----
(APP_DIR / 'schemas' / 'responses.py').write_text(textwrap.dedent('''\
    from pydantic import BaseModel
    from typing import Dict, List, Optional, Any
    from enum import Enum

    class SeverityLevel(str, Enum):
        critical = "critical"
        high = "high"
        medium = "medium"
        low = "low"

    class DecisionAction(str, Enum):
        block_and_redirect  = "block_and_redirect"
        deploy_deception    = "deploy_deception"
        increase_monitoring = "increase_monitoring"
        observe             = "observe"
        allow               = "allow"

    class ThreatDetectResponse(BaseModel):
        is_threat: bool
        confidence: float
        model_used: str
        latency_ms: float

    class AnomalyDetectResponse(BaseModel):
        is_anomaly: bool
        anomaly_score: float
        threshold: float
        method_scores: Dict[str, float]
        latency_ms: float

    class ClassifyResponse(BaseModel):
        attack_type: str
        attack_type_id: int
        confidence: float
        probabilities: Dict[str, float]
        latency_ms: float

    class ThreatIntelResponse(BaseModel):
        likely_category: str
        likely_severity: str
        confidence: float
        top_tags: List[str]
        similar_threats: List[Dict[str, Any]]
        recommendations: List[str]

    class DeceptionActionResponse(BaseModel):
        action_id: int
        action_name: str
        model_type: str
        latency_ms: float

    class PipelineResponse(BaseModel):
        flow_id: str
        timestamp: str
        is_threat: bool
        threat_confidence: float
        is_anomaly: bool
        anomaly_score: float
        attack_type: Optional[str] = None
        attack_confidence: Optional[float] = None
        severity: SeverityLevel
        decision: DecisionAction
        deception_action: Optional[str] = None
        recommendations: List[str] = []
        consensus_score: float = 0.0
        processing_time_ms: float
        agent_outputs: Dict[str, Any] = {}

    class SystemStatusResponse(BaseModel):
        overall_health: float
        n_agents: int
        n_alerts: int
        agents: Dict[str, Any]
        uptime_seconds: float
        total_events_processed: int
        recent_events: List[Dict[str, Any]] = []

    class HealthResponse(BaseModel):
        status: str
        models_loaded: Dict[str, bool]
        device: str
        uptime_seconds: float
'''), encoding='utf-8')

print('schemas/ written.')

schemas/ written.


In [13]:
# ---- app/main.py  (entry-point) ----
MAIN_PY = textwrap.dedent('''\
    """
    ACDADA — Autonomous Cyber Deception & Adaptive Defense Agent
    FastAPI backend entry-point.

    Run:
        cd backend
        uvicorn app.main:app --reload --host 0.0.0.0 --port 8000
    """
    import time, uuid, json, numpy as np, torch, torch.nn as nn
    from pathlib import Path
    from datetime import datetime, timezone
    from typing import Dict, List, Optional, Any
    from enum import Enum
    from collections import deque, Counter

    from fastapi import (
        FastAPI, HTTPException, WebSocket, WebSocketDisconnect,
        BackgroundTasks, Query,
    )
    from fastapi.middleware.cors import CORSMiddleware
    import joblib

    from app.core.config import MODELS_DIR, DEVICE, CORS_ORIGINS
    from app.schemas.requests import (
        AnalyzeRequest, ThreatDetectRequest, AnomalyDetectRequest,
        ClassifyRequest, ThreatIntelRequest, AttackProfileRequest,
        DeceptionActionRequest,
    )
    from app.schemas.responses import (
        PipelineResponse, ThreatDetectResponse, AnomalyDetectResponse,
        ClassifyResponse, DeceptionActionResponse, ThreatIntelResponse,
        HealthResponse, SystemStatusResponse, SeverityLevel, DecisionAction,
    )
    from app.models.threat_detector import ThreatDetector, load_threat_models
    from app.models.anomaly_detector import AnomalyDetector, load_anomaly_models
    from app.models.attack_classifier import AttackClassifier, load_classifier_models
    from app.models.deception_agent import DeceptionAgent, load_deception_model
    from app.models.threat_intel import ThreatIntelEngine, load_threat_intel

    # ─────────────────────────────────────────────────────────────────────────────
    # APP INIT
    # ─────────────────────────────────────────────────────────────────────────────
    app = FastAPI(
        title="ACDADA API",
        description="Autonomous Cyber Deception & Adaptive Defense Agent",
        version="1.0.0",
    )

    app.add_middleware(
        CORSMiddleware,
        allow_origins=CORS_ORIGINS,
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )

    # ─────────────────────────────────────────────────────────────────────────────
    # GLOBAL STATE
    # ─────────────────────────────────────────────────────────────────────────────
    START_TIME = time.time()
    EVENT_LOG: deque = deque(maxlen=1000)
    TOTAL_EVENTS = 0

    # Model loaders (lazy)
    threat_detector: Optional[ThreatDetector] = None
    anomaly_detector: Optional[AnomalyDetector] = None
    attack_classifier: Optional[AttackClassifier] = None
    deception_agent: Optional[DeceptionAgent] = None
    threat_intel_engine: Optional[ThreatIntelEngine] = None

    # ─────────────────────────────────────────────────────────────────────────────
    # STARTUP / SHUTDOWN
    # ─────────────────────────────────────────────────────────────────────────────
    @app.on_event("startup")
    async def startup():
        global threat_detector, anomaly_detector, attack_classifier
        global deception_agent, threat_intel_engine

        print("[ACDADA] Loading models...")
        threat_detector = load_threat_models()
        anomaly_detector = load_anomaly_models()
        attack_classifier = load_classifier_models()
        deception_agent = load_deception_model()
        threat_intel_engine = load_threat_intel()
        print("[ACDADA] Models loaded. API ready.")

    @app.on_event("shutdown")
    async def shutdown():
        print("[ACDADA] Shutting down.")

    # ─────────────────────────────────────────────────────────────────────────────
    # HELPER: LOG
    # ─────────────────────────────────────────────────────────────────────────────
    def _log_event(event: dict):
        global TOTAL_EVENTS
        event["timestamp"] = datetime.now(timezone.utc).isoformat()
        EVENT_LOG.append(event)
        TOTAL_EVENTS += 1

    # ─────────────────────────────────────────────────────────────────────────────
    # ENDPOINTS
    # ─────────────────────────────────────────────────────────────────────────────
    @app.get("/health", response_model=HealthResponse, tags=["system"])
    async def health_check():
        return HealthResponse(
            status="ok",
            models_loaded={
                "threat_detector": threat_detector is not None,
                "anomaly_detector": anomaly_detector is not None,
                "attack_classifier": attack_classifier is not None,
                "deception_agent": deception_agent is not None,
                "threat_intel_engine": threat_intel_engine is not None,
            },
            device=str(DEVICE),
            uptime_seconds=time.time() - START_TIME,
        )

    @app.get("/status", response_model=SystemStatusResponse, tags=["system"])
    async def system_status():
        return SystemStatusResponse(
            overall_health=1.0 if threat_detector else 0.0,
            n_agents=5,
            n_alerts=sum(1 for e in EVENT_LOG if e.get("is_threat")),
            agents={
                "threat_detector": {"loaded": threat_detector is not None},
                "anomaly_detector": {"loaded": anomaly_detector is not None},
                "attack_classifier": {"loaded": attack_classifier is not None},
                "deception_agent": {"loaded": deception_agent is not None},
                "threat_intel": {"loaded": threat_intel_engine is not None},
            },
            uptime_seconds=time.time() - START_TIME,
            total_events_processed=TOTAL_EVENTS,
            recent_events=list(EVENT_LOG)[-20:],
        )

    # ─── THREAT DETECTION ────────────────────────────────────────────────────────
    @app.post("/detect/threat", response_model=ThreatDetectResponse, tags=["detection"])
    async def detect_threat(req: ThreatDetectRequest):
        if threat_detector is None:
            raise HTTPException(503, "Threat detector not loaded")
        t0 = time.time()
        result = threat_detector.predict(req.features, model=req.model)
        latency = (time.time() - t0) * 1000
        _log_event({"endpoint": "detect_threat", "is_threat": result["is_threat"]})
        return ThreatDetectResponse(**result, latency_ms=latency)

    # ─── ANOMALY DETECTION ───────────────────────────────────────────────────────
    @app.post("/detect/anomaly", response_model=AnomalyDetectResponse, tags=["detection"])
    async def detect_anomaly(req: AnomalyDetectRequest):
        if anomaly_detector is None:
            raise HTTPException(503, "Anomaly detector not loaded")
        t0 = time.time()
        result = anomaly_detector.predict(req.features, threshold=req.threshold)
        latency = (time.time() - t0) * 1000
        _log_event({"endpoint": "detect_anomaly", "is_anomaly": result["is_anomaly"]})
        return AnomalyDetectResponse(**result, latency_ms=latency)

    # ─── ATTACK CLASSIFICATION ───────────────────────────────────────────────────
    @app.post("/classify", response_model=ClassifyResponse, tags=["classification"])
    async def classify_attack(req: ClassifyRequest):
        if attack_classifier is None:
            raise HTTPException(503, "Classifier not loaded")
        t0 = time.time()
        result = attack_classifier.predict(req.features, model=req.model)
        latency = (time.time() - t0) * 1000
        _log_event({"endpoint": "classify", "attack_type": result["attack_type"]})
        return ClassifyResponse(**result, latency_ms=latency)

    # ─── DECEPTION ACTION ────────────────────────────────────────────────────────
    @app.post("/deception/action", response_model=DeceptionActionResponse, tags=["deception"])
    async def get_deception_action(req: DeceptionActionRequest):
        if deception_agent is None:
            raise HTTPException(503, "Deception agent not loaded")
        t0 = time.time()
        result = deception_agent.get_action(
            threat_level=req.threat_level,
            attack_type=req.attack_type,
            is_detected=req.is_detected,
        )
        latency = (time.time() - t0) * 1000
        _log_event({"endpoint": "deception_action", "action": result["action_name"]})
        return DeceptionActionResponse(**result, latency_ms=latency)

    # ─── THREAT INTELLIGENCE ─────────────────────────────────────────────────────
    @app.post("/intel/query", response_model=ThreatIntelResponse, tags=["intelligence"])
    async def query_threat_intel(req: ThreatIntelRequest):
        if threat_intel_engine is None:
            raise HTTPException(503, "Threat intel engine not loaded")
        result = threat_intel_engine.query(req.description, k=req.top_k)
        _log_event({"endpoint": "intel_query", "category": result.get("likely_category")})
        return ThreatIntelResponse(**result)

    # ─── FULL PIPELINE ───────────────────────────────────────────────────────────
    @app.post("/analyze", response_model=PipelineResponse, tags=["pipeline"])
    async def analyze_full_pipeline(req: AnalyzeRequest):
        """
        Full ACDADA pipeline:
        1. Threat detection
        2. Anomaly detection
        3. Attack classification (if threat)
        4. Threat intel enrichment
        5. Consensus decision
        6. Deception action
        """
        t0 = time.time()
        flow_id = str(uuid.uuid4())[:12]
        agent_outputs = {}

        # 1. Threat detection
        if threat_detector:
            threat_res = threat_detector.predict(req.features, model=req.threat_model)
        else:
            threat_res = {"is_threat": False, "confidence": 0.0, "model_used": "none"}
        agent_outputs["threat_detector"] = threat_res

        # 2. Anomaly detection
        if anomaly_detector:
            anomaly_res = anomaly_detector.predict(req.features, threshold=req.anomaly_threshold)
        else:
            anomaly_res = {"is_anomaly": False, "anomaly_score": 0.0, "threshold": 0.5, "method_scores": {}}
        agent_outputs["anomaly_detector"] = anomaly_res

        is_threat = threat_res.get("is_threat", False)
        is_anomaly = anomaly_res.get("is_anomaly", False)

        # 3. Classify (if threat or anomaly)
        attack_type = None
        attack_confidence = None
        if (is_threat or is_anomaly) and attack_classifier:
            class_res = attack_classifier.predict(req.features)
            attack_type = class_res.get("attack_type")
            attack_confidence = class_res.get("confidence")
            agent_outputs["attack_classifier"] = class_res
        else:
            agent_outputs["attack_classifier"] = {"attack_type": "Benign", "confidence": 1.0}

        # 4. Threat intel
        recommendations = []
        if threat_intel_engine and (is_threat or is_anomaly):
            desc = f"{attack_type or 'Unknown'} attack with conf={threat_res.get('confidence', 0):.2f}"
            intel_res = threat_intel_engine.query(desc, k=3)
            recommendations = intel_res.get("recommendations", [])
            agent_outputs["threat_intel"] = intel_res

        # 5. Consensus decision
        threat_conf = threat_res.get("confidence", 0) if is_threat else 0
        anomaly_score = anomaly_res.get("anomaly_score", 0) if is_anomaly else 0
        class_conf = attack_confidence if attack_confidence and attack_type != "Benign" else 0
        weights = [0.4, 0.3, 0.3]
        threat_level = threat_conf * weights[0] + anomaly_score * weights[1] + class_conf * weights[2]
        n_agree = sum([1 if is_threat else 0, 1 if is_anomaly else 0, 1 if attack_type and attack_type != "Benign" else 0])
        consensus = n_agree / 3.0

        if threat_level > 0.8 or (n_agree == 3 and threat_level > 0.5):
            severity = SeverityLevel.critical
            decision = DecisionAction.block_and_redirect
        elif threat_level > 0.6 or n_agree >= 2:
            severity = SeverityLevel.high
            decision = DecisionAction.deploy_deception
        elif threat_level > 0.3:
            severity = SeverityLevel.medium
            decision = DecisionAction.increase_monitoring
        else:
            severity = SeverityLevel.low
            decision = DecisionAction.allow

        # 6. Deception action
        deception_action = None
        if deception_agent and (is_threat or is_anomaly):
            dec_res = deception_agent.get_action(
                threat_level=threat_level,
                attack_type=attack_type,
                is_detected=is_threat,
            )
            deception_action = dec_res.get("action_name")
            agent_outputs["deception_agent"] = dec_res

        processing_time = (time.time() - t0) * 1000

        event = {
            "flow_id": flow_id,
            "is_threat": is_threat,
            "is_anomaly": is_anomaly,
            "attack_type": attack_type,
            "severity": severity.value,
            "decision": decision.value,
        }
        _log_event(event)

        return PipelineResponse(
            flow_id=flow_id,
            timestamp=datetime.now(timezone.utc).isoformat(),
            is_threat=is_threat,
            threat_confidence=threat_res.get("confidence", 0),
            is_anomaly=is_anomaly,
            anomaly_score=anomaly_res.get("anomaly_score", 0),
            attack_type=attack_type,
            attack_confidence=attack_confidence,
            severity=severity,
            decision=decision,
            deception_action=deception_action,
            recommendations=recommendations,
            consensus_score=consensus,
            processing_time_ms=processing_time,
            agent_outputs=agent_outputs,
        )

    # ─── WEBSOCKET FOR STREAMING ─────────────────────────────────────────────────
    @app.websocket("/ws/events")
    async def websocket_events(websocket: WebSocket):
        await websocket.accept()
        try:
            while True:
                data = await websocket.receive_json()
                features = data.get("features", [0.0] * 50)
                # Quick pipeline
                resp = {}
                if threat_detector:
                    td = threat_detector.predict(features)
                    resp["is_threat"] = td.get("is_threat")
                    resp["threat_confidence"] = td.get("confidence")
                if anomaly_detector:
                    ad = anomaly_detector.predict(features)
                    resp["is_anomaly"] = ad.get("is_anomaly")
                    resp["anomaly_score"] = ad.get("anomaly_score")
                resp["timestamp"] = datetime.now(timezone.utc).isoformat()
                await websocket.send_json(resp)
        except WebSocketDisconnect:
            pass

    # ─── EVENT LOG ENDPOINT ──────────────────────────────────────────────────────
    @app.get("/events", tags=["system"])
    async def get_events(limit: int = Query(100, ge=1, le=1000)):
        return list(EVENT_LOG)[-limit:]

    # ─── METRICS ─────────────────────────────────────────────────────────────────
    @app.get("/metrics", tags=["system"])
    async def get_metrics():
        events = list(EVENT_LOG)
        attack_types = Counter(e.get("attack_type") for e in events if e.get("attack_type"))
        return {
            "total_events": TOTAL_EVENTS,
            "threat_rate": sum(1 for e in events if e.get("is_threat")) / max(len(events), 1),
            "anomaly_rate": sum(1 for e in events if e.get("is_anomaly")) / max(len(events), 1),
            "attack_type_distribution": dict(attack_types),
            "uptime_seconds": time.time() - START_TIME,
        }

    # ─── Run directly ────────────────────────────────────────────────────────────
    if __name__ == "__main__":
        uvicorn.run("app.main:app", host="0.0.0.0", port=8000, reload=True)
''')

(APP_DIR / 'main.py').write_text(MAIN_PY, encoding='utf-8')
print(f'app/main.py written ({len(MAIN_PY):,} chars).')

app/main.py written (14,455 chars).


---
## 7. Verify Exported Backend

In [14]:
import os

print('\n📂 backend/ tree:')
for root, dirs, files in os.walk(BACKEND_ROOT):
    level = root.replace(str(BACKEND_ROOT), '').count(os.sep)
    indent = '  ' * level
    folder = Path(root).name
    print(f'{indent}{folder}/')
    for f in sorted(files):
        size = (Path(root) / f).stat().st_size
        print(f'{indent}  {f:30s} {size:>8,} bytes')


📂 backend/ tree:
backend/
  requirements.txt                    260 bytes
  app/
    __init__.py                           0 bytes
    main.py                          17,524 bytes
    api/
      __init__.py                           0 bytes
    core/
      __init__.py                           0 bytes
      config.py                           420 bytes
    models/
      __init__.py                           0 bytes
    schemas/
      __init__.py                           0 bytes
      requests.py                         966 bytes
      responses.py                      2,051 bytes


In [15]:
# Quick syntax check — import the schemas
import importlib.util, sys

for module_name, path in [
    ('config',    APP_DIR / 'core' / 'config.py'),
    ('requests',  APP_DIR / 'schemas' / 'requests.py'),
    ('responses', APP_DIR / 'schemas' / 'responses.py'),
]:
    spec = importlib.util.spec_from_file_location(module_name, path)
    mod  = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(mod)
        print(f'  ✓ {path.relative_to(BACKEND_ROOT)} imports OK')
    except Exception as e:
        print(f'  ✗ {path.relative_to(BACKEND_ROOT)}: {e}')

  ✓ app\core\config.py imports OK
  ✓ app\schemas\requests.py imports OK
  ✓ app\schemas\responses.py imports OK


---
## 8. API Endpoint Summary

| Method | Path | Description | Dashboard Component |
|--------|------|-------------|--------------------|
| `GET`  | `/health` | System health check | Status bar |
| `GET`  | `/status` | Full system status + agent health | Home dashboard |
| `POST` | `/analyze` | Full pipeline (detect → classify → enrich → decide) | Main analyze panel |
| `POST` | `/detect/threat` | Binary threat detection only | Threat widget |
| `POST` | `/detect/anomaly` | Anomaly detection only | Anomaly widget |
| `POST` | `/classify` | Multi-class attack classification | Classification panel |
| `POST` | `/deception/action` | RL deception action selection | Deception control |
| `POST` | `/intel/query` | CTI similarity search | Intel search |
| `POST` | `/intel/profile` | Multi-indicator attack profiling | Intel profiler |
| `GET`  | `/events` | Recent event log | Event feed |
| `WS`   | `/ws/events` | Real-time event stream | Live alerts |

### Run the server:
```bash
cd backend
pip install -r requirements.txt
uvicorn app.main:app --reload --host 0.0.0.0 --port 8000
```

- Swagger docs: http://localhost:8000/docs  
- ReDoc: http://localhost:8000/redoc

In [16]:
print('✓ Notebook 10 complete.')
print('  → Backend exported to ../backend/')
print('  → Run: cd backend && uvicorn app.main:app --reload --port 8000')
print('  → Swagger: http://localhost:8000/docs')
print('\n✓ All 10 ACDADA notebooks are ready.')

✓ Notebook 10 complete.
  → Backend exported to ../backend/
  → Run: cd backend && uvicorn app.main:app --reload --port 8000
  → Swagger: http://localhost:8000/docs

✓ All 10 ACDADA notebooks are ready.
